# Module 9: RAG & LLM Applications
## 08: Streaming Responses & Real-Time Token Generators

### Theoretical Overview
Waiting for an LLM to generate 800 tokens can introduce 4-8 seconds of latency. **Streaming** leverages Server-Sent Events (SSE) to deliver tokens chunk-by-chunk as they are decoded from the model's autoregressive loop.
- **Time-to-First-Token (TTFT)**: Dramatically reduced from seconds to ~200ms.
- **Perceived Latency**: Provides immediate visual feedback to end users.
- **Token Usage Accounting**: Tracking accumulated token counts during streaming.

In [ ]:
import time
from typing import Generator

def simulate_llm_token_stream(text: str, delay_per_token: float = 0.02) -> Generator[str, None, None]:
    """Simulates an SSE streaming token generator."""
    tokens = text.split(" ")
    for idx, token in enumerate(tokens):
        time.sleep(delay_per_token)
        # Yield token with space prefix (except first)
        yield token if idx == 0 else " " + token

sample_response = (
    "Retrieval-Augmented Generation bridges the gap between static weights and enterprise knowledge bases, "
    "enabling accurate, verifiable, and dynamic question-answering pipelines."
)

print("Streaming token output: ", end="", flush=True)
accumulated_tokens = 0
start_time = time.time()
for chunk in simulate_llm_token_stream(sample_response, delay_per_token=0.015):
    print(chunk, end="", flush=True)
    accumulated_tokens += 1
total_time = time.time() - start_time

print(f"\n\n[Stream Complete] Total tokens: {accumulated_tokens} | Elapsed time: {total_time:.2f}s | Throughput: {accumulated_tokens/total_time:.1f} tokens/sec")

### Client-Side SSE Event Parser Anatomy

```python
# Real-world SSE streaming consumer loop
def consume_sse_stream(response_stream):
    for line in response_stream.iter_lines():
        if line.startswith("data: "):
            payload = line[6:]
            if payload == "[DONE]":
                break
            chunk_json = json.loads(payload)
            delta = chunk_json["choices"][0]["delta"].get("content", "")
            yield delta
```